# Introduction to Machine Learning: Supervised Learning

**Instructor:** Daniel Acuna, Ph.D.
**Position:** Associate Professor of Computer Science
**Institution:** University of Colorado Boulder

---

Lab 4: Regularization

---

# Assignment Overview

In this assignment, you'll work with the scikit-learn Diabetes regression dataset to explore
model evaluation, model selection with cross-validation, and regularization (Ridge/Lasso),
and quantify uncertainty using bootstrap. You'll produce CV curves (arrays checked by the
autograder; plots are optional and not graded).

In [2]:
# Grade Cell: Import Libraries
#
# This cell imports all necessary libraries for the assignment.
from typing import Tuple, List

import numpy as np
import pandas as pd

# Plotting is optional; imported lazily if needed
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

# Set a random state for reproducibility
RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Diabetes Dataset (10 points)

The dataset contains 10 standardized features and a continuous target. The CSV
should be available as `diabetes.csv` in this folder. If it's missing, run the
downloader script for this module.

In [3]:
# Grade Cell: Question 1
#
# Task: Load the dataset and display its first 5 rows.
#
# Instructions:
# 1. Load the 'diabetes.csv' file into a pandas DataFrame called `df`.
# 2. Use the `.head()` method to display the first 5 rows.

# Load db
df = pd.read_csv("diabetes.csv")

# top of db
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [4]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 1
assert "df" in locals(), "DataFrame 'df' not found."
assert df.shape[0] > 400, "Dataset should have more than 400 rows"
assert df.shape[1] > 10, "Dataset should have more than 10 columns"
assert "target" in df.columns, "The 'target' column is missing."
print("DataFrame loaded successfully!")
df.head()

DataFrame loaded successfully!


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 2. Prepare the Data (10 points)

Separate features `X` and target `y`.

In [5]:
# Grade Cell: Question 2
#
# Task: Prepare the data for modeling.
#
# Instructions:
# 1. Create the feature matrix `X` with all columns except `target`.
# 2. Create the target vector `y` from the `target` column.

X = df.drop(columns=['target'])
y = df['target']

In [6]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 2
assert "X" in locals() and "y" in locals(), "X and/or y are not defined."
assert X.shape[0] > 400 and X.shape[1] > 5, "X should have reasonable dimensions"
assert len(y) > 400, "y should have reasonable length"
assert "target" not in X.columns, "The 'target' column should not be in X."
print("Data preparation successful!")

Data preparation successful!


## 3. Data Splitting and Scaling (10 points)

Split into train/test sets and scale features using `StandardScaler`.

In [11]:
# Grade Cell: Question 3
#
# Task: Split and scale the data.
#
# Instructions:
# 1. Split `X` and `y` into `X_train`, `X_test`, `y_train`, and `y_test` with a `test_size` of 0.2 and `random_state=RANDOM_STATE`.
# 2. Initialize a `StandardScaler` and fit it on `X_train`.
# 3. Transform both `X_train` and `X_test` using the fitted scaler, naming them `X_train_scaled` and `X_test_scaled`.

# split up X and Y (base line)
X_train, X_test, y_train, y_test = train_test_split(
                                X, y, test_size = 0.2, random_state = RANDOM_STATE)

# set scaler, build both X train and test _scaled
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 3
assert "X_train_scaled" in locals(), "Scaled training data not found."
assert "X_test_scaled" in locals(), "Scaled test data not found."
assert X_train.shape[0] > X_test.shape[0], "Training set should be larger than test set"
assert np.allclose(
    X_train_scaled.mean(), 0, atol=1e-9
), "Scaled train mean should be ~0."
assert np.isfinite(X_train_scaled).all(), "Scaled data must be finite."
print("Data splitting and scaling successful!")

Data splitting and scaling successful!


## 4. Train OLS Baseline and Evaluate (10 points)

Train a `LinearRegression` model and evaluate on the test set using RMSE and R^2.

In [13]:
# Grade Cell: Question 4
#
# Task: Train a baseline OLS model and compute metrics.
#
# Instructions:
# 1. Fit `LinearRegression` using `X_train_scaled`, `y_train`.
# 2. Compute test RMSE (`rmse_ols_test`) and R^2 (`r2_ols_test`).

# Build lin reg and fit using X_train_scaled and y_train
ols = LinearRegression()
ols.fit(X_train_scaled, y_train)

#set up y_pred, do MSE -> RMSE  and R^2
y_pred_ols_test = ols.predict(X_test_scaled)
mse_ols_test = mean_squared_error(y_test, y_pred_ols_test)
rmse_ols_test = np.sqrt(mse_ols_test)
r2_ols_test = r2_score(y_test, y_pred_ols_test)

In [14]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 4
assert "rmse_ols_test" in locals() and "r2_ols_test" in locals(), "Missing OLS metrics."
assert isinstance(rmse_ols_test, float) and isinstance(
    r2_ols_test, float
), "Metrics should be floats"
assert rmse_ols_test > 0, "RMSE must be positive"
assert -1 <= r2_ols_test <= 1, "R^2 must be between -1 and 1"
print(f"OLS RMSE: {rmse_ols_test:.3f}, R^2: {r2_ols_test:.3f}")

OLS RMSE: 53.853, R^2: 0.453


## 5. 5-fold CV for OLS (10 points)

Use K-fold cross-validation on the training set to estimate OLS generalization error.

In [15]:
# Grade Cell: Question 5
#
# Task: Compute CV RMSE mean and std for OLS on the training split.
#
# Instructions:
# - Use `cross_val_score` with `scoring='neg_mean_squared_error'` and `KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`.
# - Store `rmse_ols_cv_mean` and `rmse_ols_cv_std` (floats).


cv = KFold(n_splits = 5, shuffle = True, random_state = RANDOM_STATE)
neg_mse_scores = cross_val_score(
    LinearRegression(), X_train_scaled, y_train
    ,scoring = 'neg_mean_squared_error', cv = cv
)
rmse_folds = np.sqrt(-neg_mse_scores)
rmse_ols_cv_mean = rmse_folds.mean()
rmse_ols_cv_std = rmse_folds.std()




In [16]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 5
assert (
    "rmse_ols_cv_mean" in locals() and "rmse_ols_cv_std" in locals()
), "Missing CV stats."
assert isinstance(rmse_ols_cv_mean, float) and isinstance(
    rmse_ols_cv_std, float
), "CV stats should be floats"
assert rmse_ols_cv_mean > 0, "Mean RMSE must be positive"
assert rmse_ols_cv_std >= 0, "Std must be non-negative"
print(f"OLS CV RMSE mean: {rmse_ols_cv_mean:.3f}, std: {rmse_ols_cv_std:.3f}")

OLS CV RMSE mean: 55.395, std: 2.361


## 6. Ridge Regression with Cross-Validation (10 points)

Use `RidgeCV` over a log-spaced grid to select the best alpha, then evaluate on the test set.

In [19]:
# Grade Cell: Question 6
#
# Task: Train RidgeCV and compute test metrics.
#
# Instructions:
# - Use `ridge_alphas = np.logspace(-3, 3, 50)`
# - Fit on `X_train_scaled`, `y_train`
# - Store `ridge_best_alpha`, `rmse_ridge_test`, `r2_ridge_test`

ridge_alphas = np.logspace(-3, 3, 50)
ridge_cv = RidgeCV(alphas = ridge_alphas, cv = cv, scoring = 'neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)
ridge_best_alpha = ridge_cv.alpha_

y_pred_ridge_test = ridge_cv.predict(X_test_scaled)
rmse_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test))
r2_ridge_test = r2_score(y_test, y_pred_ridge_test)



In [20]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 6
assert "ridge_best_alpha" in locals(), "ridge_best_alpha not found."
assert ridge_best_alpha > 0, "Alpha must be positive"
assert isinstance(rmse_ridge_test, float) and isinstance(
    r2_ridge_test, float
), "Metrics should be floats"
assert rmse_ridge_test > 0 and -1 <= r2_ridge_test <= 1, "Metrics out of valid range"
print(f"Ridge best alpha: {ridge_best_alpha:.5f}")

Ridge best alpha: 0.86851


## 7. Lasso with Cross-Validation (10 points)

Use `LassoCV` to select alpha and evaluate on the test set. Also count non-zero coefficients.

In [27]:
# Grade Cell: Question 7
#
# Task: Train LassoCV and compute metrics.
#
# Instructions:
# - Use `lasso_alphas = np.logspace(-3, 1, 50)`, `cv=5`, `random_state=RANDOM_STATE`, `max_iter=10000`
# - Store `lasso_best_alpha`, `rmse_lasso_test`, `r2_lasso_test`, `n_nonzero_lasso`

lasso_alphas = np.logspace(-3, 3, 50)
lasso_cv = LassoCV(alphas = lasso_alphas, cv = 5, random_state = RANDOM_STATE, max_iter = 10000)
lasso_cv.fit(X_train_scaled, y_train)
lasso_best_alpha = lasso_cv.alpha_

y_pred_lasso_test = lasso_cv.predict(X_test_scaled)
rmse_lasso_test = np.sqrt(mean_squared_error(y_test, y_pred_lasso_test))
r2_lasso_test = r2_score(y_test, y_pred_lasso_test)
n_nonzero_lasso = int(np.sum(lasso_cv.coef_ != 0))



In [28]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 7
assert (
    "lasso_best_alpha" in locals() and "n_nonzero_lasso" in locals()
), "Missing Lasso outputs"
assert lasso_best_alpha > 0, "Alpha must be positive"
assert isinstance(n_nonzero_lasso, int), "Non-zero count should be an integer"
assert n_nonzero_lasso >= 0, "Non-zero count cannot be negative"
assert isinstance(rmse_lasso_test, float) and isinstance(
    r2_lasso_test, float
), "Metrics should be floats"
print(f"Lasso best alpha: {lasso_best_alpha:.5f}, non-zero: {n_nonzero_lasso}")

Lasso best alpha: 1.52642, non-zero: 8


## 8. Cross-Validation Curves (10 points)

expose arrays for plotting CV error vs alpha. Plots are optional and not graded.

In [30]:
# Grade Cell: Question 8
#
# Task: Expose arrays for CV curves.
#
# Instructions:
# - For Ridge: use `ridge_alphas` and compute mean MSE over folds from `ridge_cv.cv_values_` (if available).
# - For Lasso: use `lasso_alphas` and `lasso_cv.mse_path_` (mean across folds).

from sklearn.linear_model import Ridge

ridge_cv_mse_mean = []
for alpha in ridge_alphas:
    scores = cross_val_score(
        Ridge(alpha=alpha), X_train_scaled, y_train
        ,scoring = 'neg_mean_squared_error', cv=cv
    )
    ridge_cv_mse_mean.append(-scores.mean())
ridge_cv_mse_mean = np.array(ridge_cv_mse_mean)

lasso_cv_mse_mean = lasso_cv.mse_path_.mean(axis=1)

In [31]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 8
assert (
    "ridge_cv_mse_mean" in locals() and "lasso_cv_mse_mean" in locals()
), "Missing CV arrays."
assert isinstance(ridge_cv_mse_mean, np.ndarray) and isinstance(
    lasso_cv_mse_mean, np.ndarray
), "Arrays should be numpy arrays"
assert (
    len(ridge_cv_mse_mean) > 0 and len(lasso_cv_mse_mean) > 0
), "Arrays should not be empty"
assert np.isfinite(lasso_cv_mse_mean).all(), "Lasso CV MSE must be finite."
print("CV curve arrays prepared.")

CV curve arrays prepared.


## 9. Bootstrap Coefficient Uncertainty (10 points)

Use bootstrap on the training set to estimate uncertainty of OLS coefficients.
Return the bootstrap distribution (B x p) and 95% CIs.

In [34]:
# Grade Cell: Question 9
#
# Task: Implement a bootstrap procedure for OLS coefficients.
#
# Instructions:
# - Implement a function `bootstrap_ols_coefficients` that:
#   * draws B bootstrap samples of the training set (with replacement)
#   * fits OLS on each sample using scaled features
#   * stores coefficient vectors
#   * returns (coef_bootstrap_df, coef_ci_95)
# - Use B=200, RANDOM_STATE


def bootstrap_ols_coefs(X, y, B = 200, random_state = RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
    coefs = []
    for _ in range(B):
        idx = rng.choice(n, size = n, replace = True)
        model = LinearRegression()
        model.fit(X[idx], y_arr[idx])
        coefs.append(model.coef_)
    coef_bootstrap_df = pd.DataFrame(coefs, columns = X_train.columns)
    coef_ci_95 = coef_bootstrap_df.quantile([0.025, 0.975]).T
    coef_ci_95.columns = ['lower', 'upper']
    return coef_bootstrap_df, coef_ci_95

coef_bootstrap_df, coef_ci_95 = bootstrap_ols_coefs(X_train_scaled, y_train, B = 200, random_state = RANDOM_STATE)

        
        

In [35]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 9
assert (
    "coef_bootstrap_df" in locals() and "coef_ci_95" in locals()
), "Bootstrap outputs missing."
assert isinstance(
    coef_bootstrap_df, pd.DataFrame
), "Bootstrap output should be DataFrame"
assert coef_bootstrap_df.shape[0] > 50, "Bootstrap should have many samples"
assert coef_bootstrap_df.shape[1] > 5, "Bootstrap should cover all features"
assert {"lower", "upper"}.issubset(coef_ci_95.columns), "CI must have lower/upper."
assert np.isfinite(
    coef_bootstrap_df.to_numpy()
).all(), "Bootstrap coefficients must be finite."
print("Bootstrap coefficients computed.")

Bootstrap coefficients computed.


## 10. Bootstrap OOB Performance (10 points)

Estimate performance uncertainty using bootstrap out-of-bag RMSE on the training set.

In [41]:
# Grade Cell: Question 10
#
# Task: Implement Bootstrap OOB RMSE for OLS.
#
# Instructions:
# - Implement `bootstrap_oob_rmse_ols` that returns (rmse_oob_mean, rmse_oob_ci95)
# - Use B=200, RANDOM_STATE


def bootstrap_oob_rmse_ols(X, y, B = 200, random_state = RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    y_arr = y.values if hasattr(y, 'values') else np.asarray(y)
    oob_rmses = []
    for _ in range(B):
        idx = rng.choice(n, size = n, replace = True)
        oob_idx = np.setdiff1d(np.arange(n), idx)
        if len(oob_idx) == 0:
            continue
        model = LinearRegression()
        model.fit(X[idx], y_arr[idx])
        preds = model.predict(X[oob_idx])
        rmse = np.sqrt(mean_squared_error(y_arr[oob_idx], preds))
        oob_rmses.append(rmse)
    oob_rmses = np.array(oob_rmses)
    rmse_oob_mean = float(oob_rmses.mean())
    rmse_oob_ci95 = (float(np.percentile(oob_rmses, 2.5)), float(np.percentile(oob_rmses, 97.5)))
    return rmse_oob_mean, rmse_oob_ci95

rmse_oob_mean, rmse_oob_ci95 = bootstrap_oob_rmse_ols(X_train_scaled, y_train, B = 200, random_state = RANDOM_STATE)
                     



In [42]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 10
assert (
    "rmse_oob_mean" in locals() and "rmse_oob_ci95" in locals()
), "Missing OOB metrics."
assert (
    isinstance(rmse_oob_ci95, tuple) and len(rmse_oob_ci95) == 2
), "CI must be a tuple (lower, upper)."
assert isinstance(rmse_oob_mean, float), "OOB mean should be a float"
assert rmse_oob_mean > 0, "OOB RMSE must be positive"
assert rmse_oob_ci95[0] < rmse_oob_mean < rmse_oob_ci95[1], "Mean must lie within CI."
print(
    f"OOB RMSE mean: {rmse_oob_mean:.3f}, 95% CI: [{rmse_oob_ci95[0]:.3f}, {rmse_oob_ci95[1]:.3f}]"
)

OOB RMSE mean: 55.841, 95% CI: [50.778, 60.683]


## Next Steps

Congratulations on completing the assignment! Before submitting:

1. Make sure all your cells run without errors.
2. Ensure you've answered all parts of each question.
3. If any autograder tests fail, revisit your answers.
